In [1]:
import pandas as pd
import yaml
import io
import openpyxl
import csv

In [2]:
excel_file = 'DV.xlsx'

xl_file = pd.ExcelFile(excel_file)
sheet_names = xl_file.sheet_names
print(f"Các sheet trong file: {sheet_names}")

sheet_list = ['Production, Raw Mat',
'Energy, GHG',
'Water',
'Waste',
'Emission']

for sheet in sheet_names:
    if sheet in sheet_list:
        df = pd.read_excel(excel_file, sheet_name=sheet)
        print(f"Sheet: {sheet}")
        print(f"Kích thước: {df.shape[0]} dòng, {df.shape[1]} cột")
        print(df)
    print("\n")

Các sheet trong file: ['Business', 'Scope', 'Production, Raw Mat', 'Energy, GHG', 'Energy, GHG (2)', 'Water', 'Waste', 'Emission', 'CO2 SAY PHUN']




Sheet: Production, Raw Mat
Kích thước: 20 dòng, 19 cột
                                           Unnamed: 0 Unnamed: 1 Unnamed: 2  \
0                    Production (DV Tile + YB Powder)       unit        Jan   
1                  Production (DV Tile) (Sản phẩm ĐV)        ton   9609.931   
2       Production (YB Powder) (bột bán cho Yên Bình)        ton          0   
3                          Total Production (DV + YB)        ton   9609.931   
4                                                 NaN        NaN        NaN   
5                                        Raw Material       unit        Jan   
6                          Limestone/ Đá vôi (canxit)        ton    1185.21   
7                                       Clay/ Đất sét        ton    6280.95   
8                                           Sand/ Cát        ton        NaN   
9   

Energy, GHG

In [3]:
# sheet 'Energy, GHG'
df_energy = pd.read_excel(excel_file, sheet_name='Energy, GHG')
print(df_energy.head(10))

                                     Unnamed: 0 Unnamed: 1 Unnamed: 2  \
0                            Energy Consumption       unit        Jan   
1                            Coal/ Than 5A (DV)        Ton   1233.522   
2                          Coal/ Than 4A.1 (DV)        Ton        443   
3  Coal/ Than import coal (DV) (than nhập khẩu)        Ton          0   
4   Than cám qua sàng / Than 5A under screen DV        Ton        114   
5                            Coal/ Than 5A (YB)        Ton          0   
6                          Coal/ Than 4A.1 (YB)        Ton          0   
7  Coal/ Than import coal (YB) (than nhập khẩu)        Ton          0   
8   Than cám qua sàng / Than 5A under screen YB        Ton          0   
9                         Coal/ Than 5A (DV+YB)        Ton   1233.522   

  Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  \
0        Feb        Mar        Apr        May        Jun        Jul   
1        983   2414.661   2198.795   2197.301    2170.

In [4]:
def find_sections(df, section_names):
    section_markers = {}
    for idx, row in df.iterrows():
        row_str = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ''
        for name in section_names:
            if row_str == name:
                section_markers[name] = idx
    return section_markers

In [ ]:
energy_df = df_energy.reset_index(drop=True)

section_names = ['Energy Consumption', 'Greenhouse Gas Emission', 'Conversion Factor', 'Energy Cost']
section_markers = find_sections(energy_df, section_names)

sorted_sections = sorted([(idx, s) for s, idx in section_markers.items() if idx is not None])

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(energy_df)
    df_part = energy_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(10).to_string())

    

Energy Consumption
65 hàng x 22 cột
0                            Energy Consumption unit       Jan     Feb       Mar       Apr       May      Jun       Jul       Aug       Sep       Oct      Nov         Dec  YTD        Q1        Q2        Q3          Q4  NaN  NaN  NaN
0                            Coal/ Than 5A (DV)  Ton  1233.522     983  2414.661  2198.795  2197.301  2170.99  2442.916  2148.882  1718.913  1753.955   1649.3    2061.215  NaN  4631.183  6567.086  6310.711     5464.47  NaN  NaN  NaN
1                          Coal/ Than 4A.1 (DV)  Ton       443  251.25   543.069    88.335    774.19  319.418    508.88   630.609   444.559      18.9   113.76     369.092  NaN  1237.319  1181.943  1584.048     501.752  NaN  NaN  NaN
2  Coal/ Than import coal (DV) (than nhập khẩu)  Ton         0   92.48   430.117   722.734         0   434.63    375.27     77.21         0    550.88  352.255     533.023  NaN   522.597  1157.364    452.48    1436.158  NaN  NaN  NaN
3   Than cám qua sàng / Than 5A 

In [ ]:
import re

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(energy_df)
    df_part = energy_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(10).to_string())

    safe_section = re.sub(r'[^A-Za-z0-9_]+', '_', section)
    df_part_clean.to_csv(f"{safe_section}.csv", index=False)

Energy Consumption
65 hàng x 22 cột
0                            Energy Consumption unit       Jan     Feb       Mar       Apr       May      Jun       Jul       Aug       Sep       Oct      Nov         Dec  YTD        Q1        Q2        Q3          Q4  NaN  NaN  NaN
0                            Coal/ Than 5A (DV)  Ton  1233.522     983  2414.661  2198.795  2197.301  2170.99  2442.916  2148.882  1718.913  1753.955   1649.3    2061.215  NaN  4631.183  6567.086  6310.711     5464.47  NaN  NaN  NaN
1                          Coal/ Than 4A.1 (DV)  Ton       443  251.25   543.069    88.335    774.19  319.418    508.88   630.609   444.559      18.9   113.76     369.092  NaN  1237.319  1181.943  1584.048     501.752  NaN  NaN  NaN
2  Coal/ Than import coal (DV) (than nhập khẩu)  Ton         0   92.48   430.117   722.734         0   434.63    375.27     77.21         0    550.88  352.255     533.023  NaN   522.597  1157.364    452.48    1436.158  NaN  NaN  NaN
3   Than cám qua sàng / Than 5A 

Production, Raw Mat

In [83]:
# sheet 'Energy, GHG'
df_production = pd.read_excel(excel_file, sheet_name='Production, Raw Mat')
print(df_production.head(10))

                                      Unnamed: 0 Unnamed: 1 Unnamed: 2  \
0               Production (DV Tile + YB Powder)       unit        Jan   
1             Production (DV Tile) (Sản phẩm ĐV)        ton   9609.931   
2  Production (YB Powder) (bột bán cho Yên Bình)        ton          0   
3                     Total Production (DV + YB)        ton   9609.931   
4                                            NaN        NaN        NaN   
5                                   Raw Material       unit        Jan   
6                     Limestone/ Đá vôi (canxit)        ton    1185.21   
7                                  Clay/ Đất sét        ton    6280.95   
8                                      Sand/ Cát        ton        NaN   
9                              Gypsum/ Thạch cao        ton        NaN   

    Unnamed: 3    Unnamed: 4    Unnamed: 5    Unnamed: 6    Unnamed: 7  \
0          Feb           Mar           Apr           May           Jun   
1  6310.080933  17732.973467  15795.4

In [84]:
pro_df = df_production.reset_index(drop=True)
section_names = ['Production (DV Tile + YB Powder)', 'Raw Material']
section_markers = find_sections(pro_df, section_names)
sorted_sections = sorted([(section_markers[s], s) for s in section_names if s in section_markers])

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(pro_df)
    df_part = pro_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(10).to_string())

Production (DV Tile + YB Powder)
4 hàng x 19 cột
0               Production (DV Tile + YB Powder) unit       Jan          Feb           Mar           Apr           May           Jun           Jul           Aug           Sep           Oct          Nov           Dec            YTD          Q1            Q2            Q3           Q4
0             Production (DV Tile) (Sản phẩm ĐV)  ton  9609.931  6310.080933  17732.973467  15795.455367  16091.799767  16408.015833  18454.323805  16179.630248  12136.760367  12464.841216  12324.14555  15580.268823  169088.226375  33652.9854  48295.270967  46770.714419  40369.25559
1  Production (YB Powder) (bột bán cho Yên Bình)  ton         0            0             0             0             0           NaN           NaN           NaN           NaN           NaN          NaN      423.6674       423.6674           0             0             0     423.6674
2                     Total Production (DV + YB)  ton  9609.931  6310.080933  17732.973467  15795.4

Water

In [89]:
water_df = pd.read_excel(excel_file, sheet_name='Water')
print(water_df.head(10))

                                     Unnamed: 0 Unnamed: 1 Unnamed: 2  \
0                              Water Withdrawal       unit        Jan   
1          Surface water (Lake) (DV) (Nước mặt)         m3       4987   
2          Surface water (Lake) (YB) (Nước Mặt)         m3          0   
3  Surface water (Lake) (DV+YB) (tổng nước mặt)         m3       4987   
4                     Ground water (wells) (DV)         m3          0   
5                     Tap wate (DV) (Nước sạch)         m3        658   
6                   Total Water Withdrawal (DV)         m3       5645   
7                   Total Water Withdrawal (YB)         m3          0   
8                Total Water Withdrawal (DV+YB)         m3       5645   
9                Specific Water Withdrawal (DV)     m3/ton   0.587413   

  Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  \
0        Feb        Mar        Apr        May        Jun        Jul   
1       3058       6675       6392       6393       62

In [90]:
water_df = water_df.reset_index(drop=True)
section_names = ['Water Withdrawal', 'Water Recycle']
section_markers = find_sections(water_df, section_names)
sorted_sections = sorted([(section_markers[s], s) for s in section_names if s in section_markers])
for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(water_df)
    df_part = water_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(10).to_string())

Water Withdrawal
12 hàng x 19 cột
0                              Water Withdrawal    unit       Jan       Feb       Mar       Apr       May       Jun       Jul       Aug       Sep       Oct       Nov       Dec     YTD      Q1     Q2     Q3     Q4
0          Surface water (Lake) (DV) (Nước mặt)      m3      4987      3058      6675      6392      6393      6254      6934      6241      5438      6081      6741      7894   73088   14720  19039  18613  20716
1          Surface water (Lake) (YB) (Nước Mặt)      m3         0         0         0         0         0         0         0         0         0         0         0         0       0       0      0      0      0
2  Surface water (Lake) (DV+YB) (tổng nước mặt)      m3      4987      3058      6675      6392      6393      6254      6934      6241      5438      6081      6741      7894   73088   14720  19039  18613  20716
3                     Ground water (wells) (DV)      m3         0         0         0         0         0         

#Waste

In [103]:
waste_df = pd.read_excel(excel_file, sheet_name='Waste')
print(waste_df.head(10))

                                          Unnamed: 0               Unnamed: 1  \
0          Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI                      Jan   
1                                                NaN  Incoming Stock/ Tồn đầu   
2                                                NaN                      NaN   
3  Contaminated Cloth/ Bao gồm giẻ lau, gang tay ...                   0.3472   
4  Used oil/ Tất cả các loại dầu đã qua sử dụng thải                   5.8526   
5  Package contaminated / Vỏ thùng sơn, vỏ thùng ...                    2.413   
6  Electronic equipment / Các thiết bị điện thải ...                   0.1484   
7  Insulation/ Vật liệu cách nhiệt (amiăng thải, ...                   1.6998   
8                                  Đất dính hóa chất                   0.2316   
9                     Cặn rắn từ quá trình sử lý men                     0.04   

                             Unnamed: 2                  Unnamed: 3  \
0                                   N

In [104]:
df_waste = waste_df.reset_index(drop=True)
df_waste.info()
df_waste.head(10)
section_names = ['Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI', 'Non Hazardous Waste (ton)/ CHẤT THẢI THÔNG THƯỜNG']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 67 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   24 non-null     object 
 1   Unnamed: 1   24 non-null     object 
 2   Unnamed: 2   24 non-null     object 
 3   Unnamed: 3   10 non-null     object 
 4   Unnamed: 4   4 non-null      object 
 5   Unnamed: 5   6 non-null      object 
 6   Unnamed: 6   24 non-null     object 
 7   Unnamed: 7   24 non-null     object 
 8   Unnamed: 8   8 non-null      object 
 9   Unnamed: 9   6 non-null      object 
 10  Unnamed: 10  6 non-null      object 
 11  Unnamed: 11  24 non-null     object 
 12  Unnamed: 12  23 non-null     object 
 13  Unnamed: 13  12 non-null     object 
 14  Unnamed: 14  4 non-null      object 
 15  Unnamed: 15  6 non-null      object 
 16  Unnamed: 16  24 non-null     object 
 17  Unnamed: 17  23 non-null     object 
 18  Unnamed: 18  6 non-null      object 
 19  Unnamed: 1

In [ ]:
section_names = [
    'Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI',
    'Non Hazardous Waste (ton)/ CHẤT THẢI THÔNG THƯỜNG'
]
section_markers = find_sections(waste_df, section_names)
sorted_sections = sorted([(section_markers[s], s) for s in section_names if s in section_markers])

all_flat = []

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(waste_df)
    df_part = waste_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(3).to_string())

    data = df_part_clean.iloc[3:].reset_index(drop=True)
    data.columns = [f"col_{i}" for i in range(data.shape[1])]

    for idx, row in data.iterrows():
        waste_category_en = row['col_0']
        for m, month in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
            base = 1 + m * 5
            record = {
                'section': section,
                'waste_category_en': waste_category_en,
                'report_month': month,
                'incoming_stock': row.get(f'col_{base}'),
                'waste_generated': row.get(f'col_{base+1}'),
                'reuse_recycle': row.get(f'col_{base+2}'),
                'incineration': row.get(f'col_{base+3}'),
                'landfill': row.get(f'col_{base+4}'),
            }
            all_flat.append(record)

df_flat = pd.DataFrame(all_flat)
print(df_flat.head(10))

Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI
14 hàng x 67 cột
0                              Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI                      Jan                                   NaN                         NaN                          NaN                           NaN                      Feb                                   NaN                         NaN                          NaN                           NaN                      Mar                                   NaN                         NaN                          NaN                           NaN                      Apr                                   NaN                         NaN                          NaN                           NaN                      May                                   NaN                         NaN                          NaN                           NaN                      Jun                                   NaN                         NaN                          Na

Emission

| STT | Tên cột (Field Name)         | Kiểu dữ liệu   | Ý nghĩa & Ví dụ từ ảnh                                   | Ghi chú                                    |
|-----|------------------------------|---------------|---------------------------------------------------------|--------------------------------------------|
| 1   | stack_no                     | String        | Tên ống khói (Sấy phun 1, Sấy phun 2...)                | Khóa chính để phân loại nguồn thải          |
| 2   | process                      | String        | Quy trình sản xuất liên quan                             |                                            |
| 3   | diameter                     | Float         | Đường kính ống khói (m)                                 | Thông số kỹ thuật cố định                   |
| 4   | sampling_month_year          | String/Date   | Tháng/Năm lấy mẫu (Ví dụ: 2/12/2019)                    | Cột quan trọng để phân biệt các đợt đo      |
| 5   | stack_temperature            | Float         | Nhiệt độ ống khói (°C)                                  |                                            |
| 6   | oxygen_pct                   | Float         | % Oxygen trong khí thải                                  |                                            |
| 7   | gas_velocity                 | Float         | Vận tốc khí (m/s)                                       |                                            |
| 8   | flow_rate                    | Float         | Lưu lượng khí (m3/min)                                  |                                            |
| 9   | dust_emission_actual_o2      | Float         | Kết quả đo bụi thực tế (mg/m3)                          |                                            |
| 10  | dust_emission_rate           | Float         | Tỉ lệ bụi phát thải (kg/hr)                             |                                            |
| 11  | operation_hour               | Float         | Thời gian hoạt động (hr)                                |                                            |
| 12  | dust_emission_total          | Float         | Tổng lượng bụi phát thải (kg)                           | Kết quả của (Cột 10 * Cột 11)              |

In [111]:
emis_df = pd.read_excel(excel_file, sheet_name='Emission')
print(emis_df.head(10))
emis_df = emis_df.reset_index(drop=True)
section_names = ['Dust/ Bụi']

                   Unnamed: 0 Unnamed: 1                     Unnamed: 2  \
0                   Dust/ Bụi        NaN                            NaN   
1                   Stack No.    Process  Diameter/ Đường kính ống khói   
2                         NaN        NaN                              m   
3                  Sấy phun 1        NaN                            1.3   
4                  Sấy phun 2        NaN                            1.3   
5  Ống khói lò nung xương 1,2        NaN                            0.8   
6  Ống khói lò nung xương 3,4        NaN                            0.8   
7  Ống khói lò nung xương 5,6        NaN                            0.8   
8   Ống khói lò nung men số 1        NaN                            0.8   
9   Ống khói lò nung men số 2        NaN                            0.8   

                Unnamed: 3           Unnamed: 4 Unnamed: 5    Unnamed: 6  \
0                      Jan                  NaN        NaN           NaN   
1  Sampling \n(Month/Y

In [126]:
section_names = [
    'Dust/ Bụi',
   
]
section_markers = find_sections(emis_df, section_names)
sorted_sections = sorted([(section_markers[s], s) for s in section_names if s in section_markers])

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
n_month_cols = 9 

all_flat = []

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(emis_df)
    df_part = emis_df.iloc[start_idx:end_idx].copy()
    header_row = 1
    data_start_row = 3
    columns = df_part.iloc[header_row].values
    df_data = pd.DataFrame(df_part.iloc[data_start_row:].values, columns=columns)

    for idx, row in df_data.iterrows():
        stack_no = row['Stack No.']
        process = row['Process']
        diameter = pd.to_numeric(row['Diameter/ Đường kính ống khói'], errors='coerce')
        for m, month in enumerate(months):
            base = 3 + m * n_month_cols

            if base + n_month_cols - 1 >= len(row):
                continue
            sampling_month_year = row.iloc[base]
            stack_temperature = pd.to_numeric(row.iloc[base + 1], errors='coerce')
            oxygen_pct = pd.to_numeric(row.iloc[base + 2], errors='coerce')
            gas_velocity = pd.to_numeric(row.iloc[base + 3], errors='coerce')
            flow_rate = pd.to_numeric(row.iloc[base + 4], errors='coerce')
            dust_emission_actual_o2 = pd.to_numeric(row.iloc[base + 5], errors='coerce')
            dust_emission_rate = pd.to_numeric(row.iloc[base + 6], errors='coerce')
            operation_hour = pd.to_numeric(row.iloc[base + 7], errors='coerce')
            dust_emission_total = pd.to_numeric(row.iloc[base + 8], errors='coerce')

            if pd.isna(sampling_month_year):
                continue
            record = {
                'section': section,
                'stack_no': stack_no,
                'process': process,
                'diameter': diameter,
                'sampling_month_year': sampling_month_year,
                'stack_temperature': stack_temperature,
                'oxygen_pct': oxygen_pct,
                'gas_velocity': gas_velocity,
                'flow_rate': flow_rate,
                'dust_emission_actual_o2': dust_emission_actual_o2,
                'dust_emission_rate': dust_emission_rate,
                'operation_hour': operation_hour,
                'dust_emission_total': dust_emission_total,
                'report_month': month
            }
            all_flat.append(record)

df_emission_flat = pd.DataFrame(all_flat)
print(df_emission_flat.head(10))

     section    stack_no process  diameter sampling_month_year  \
0  Dust/ Bụi  Sấy phun 1     NaN       1.3           2/12/2019   
1  Dust/ Bụi  Sấy phun 1     NaN       1.3           2/12/2019   
2  Dust/ Bụi  Sấy phun 1     NaN       1.3           2/12/2019   
3  Dust/ Bụi  Sấy phun 1     NaN       1.3            3/4/2020   
4  Dust/ Bụi  Sấy phun 1     NaN       1.3            3/4/2020   
5  Dust/ Bụi  Sấy phun 1     NaN       1.3            8/6/2020   
6  Dust/ Bụi  Sấy phun 1     NaN       1.3            8/6/2020   
7  Dust/ Bụi  Sấy phun 1     NaN       1.3            8/6/2020   
8  Dust/ Bụi  Sấy phun 1     NaN       1.3            8/6/2020   
9  Dust/ Bụi  Sấy phun 1     NaN       1.3           20/8/2020   

   stack_temperature  oxygen_pct  gas_velocity    flow_rate  \
0               84.3         NaN           NaN  1416.666667   
1               84.3         NaN           NaN  1416.666667   
2               84.3         NaN           NaN  1416.666667   
3               82.3 

| STT | Tên cột (Field Name) | Kiểu dữ liệu | Ý nghĩa & Ví dụ từ ảnh | Ghi chú |
| :--- | :--- | :--- | :--- | :--- |
| 1 | `stack_no` | String | Tên ống khói (Ví dụ: Stack 01) | Khóa chính để định danh |
| 2 | `process` | String | Quy trình sản xuất | |
| 3 | `diameter` | Float | Đường kính ống khói (m) | |
| 4 | `report_month` | String | Tháng báo cáo (Jan, Feb, Mar...) | Cột dùng để phân nhóm/lọc dữ liệu |
| 5 | `sampling_date` | String/Date | Ngày lấy mẫu cụ thể (MM/YYYY) | Đối chiếu với phiếu quan trắc |
| 6 | `stack_temperature` | Float | Nhiệt độ ống khói (°C) | |
| 7 | `oxygen_pct` | Float | % Oxygen | |
| 8 | `gas_velocity` | Float | Vận tốc khí (m/s) | |
| 9 | `flow_rate` | Float | Lưu lượng khí ($m^3/min$) | |
| 10 | `sox_emission_actual_o2` | Float | Kết quả đo SOx thực tế ($mg/m^3$) | |
| 11 | `sox_emission_rate` | Float | Tỉ lệ phát thải SOx ($kg/hr$) | |
| 12 | `operation_hour` | Float | Thời gian hoạt động (hr) | |
| 13 | `sox_emission_total` | Float | Tổng khí thải SOx (kg) | Kết quả cuối của khối màu xám |

In [129]:
section_names = [
    'Nox',
    'Sox/ Khí Sox'
]

In [131]:

section_markers = find_sections(emis_df, section_names)
sorted_sections = sorted([(section_markers[s], s) for s in section_names if s in section_markers])

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
n_month_cols = 9

all_flat = []

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(emis_df)
    df_part = emis_df.iloc[start_idx:end_idx].copy()
    header_row = 1
    data_start_row = 3
    columns = df_part.iloc[header_row].values
    df_data = pd.DataFrame(df_part.iloc[data_start_row:].values, columns=columns)

    for idx, row in df_data.iterrows():
        stack_no = row['Stack No.']
        process = row['Process']
   
        for m, month in enumerate(months):
            base = 3 + m * n_month_cols
            if base + n_month_cols - 1 >= len(row):
                continue
            sampling_date = row.iloc[base]
            stack_temperature = pd.to_numeric(row.iloc[base + 1], errors='coerce')
            oxygen_pct = pd.to_numeric(row.iloc[base + 2], errors='coerce')
            gas_velocity = pd.to_numeric(row.iloc[base + 3], errors='coerce')
            flow_rate = pd.to_numeric(row.iloc[base + 4], errors='coerce')
            nox_emission_actual_o2 = pd.to_numeric(row.iloc[base + 5], errors='coerce')
            nox_emission_rate = pd.to_numeric(row.iloc[base + 6], errors='coerce')
            operation_hour = pd.to_numeric(row.iloc[base + 7], errors='coerce')
            nox_emission_total = pd.to_numeric(row.iloc[base + 8], errors='coerce')

            if pd.isna(sampling_date):
                continue
            record = {
                'section': section,
                'stack_no': stack_no,
                'process': process,
                'diameter': diameter,
                'report_month': month,
                'sampling_date': sampling_date,
                'stack_temperature': stack_temperature,
                'oxygen_pct': oxygen_pct,
                'gas_velocity': gas_velocity,
                'flow_rate': flow_rate,
                'nox_emission_actual_o2': nox_emission_actual_o2,
                'nox_emission_rate': nox_emission_rate,
                'operation_hour': operation_hour,
                'nox_emission_total': nox_emission_total
            }
            all_flat.append(record)

df_nox_flat = pd.DataFrame(all_flat)
print(df_nox_flat.head(10))

        section    stack_no  process  diameter report_month sampling_date  \
0  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          Jan     2/12/2019   
1  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          Feb     2/12/2019   
2  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          Mar     2/12/2019   
3  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          Apr      3/4/2020   
4  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          May      3/4/2020   
5  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          Jun      8/6/2020   
6  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          Jul      8/6/2020   
7  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          Aug      8/6/2020   
8  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          Sep      8/6/2020   
9  Sox/ Khí Sox  Sấy phun 1      NaN       NaN          Oct     20/8/2020   

   stack_temperature  oxygen_pct  gas_velocity    flow_rate  \
0               84.3         NaN           NaN  1416.666667   
1               84.3      